# 🚀 Lesson 25: Fine-Tuning BERT for Sequence Classification

**Advanced Step-by-Step Interactive Notebook** with clear architectural context, code logic, and step explanations.


### 🔹 Step 1: Execution Block

**Purpose**: Import required libraries and frameworks (e.g. PyTorch, NumPy, Sklearn, Hugging Face).

- Sets up the execution environment, random seeds, and GPU/MPS device acceleration if available.


In [ ]:
# ==============================================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score

from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup
)

from huggingface_hub import login

# ------------------------------------------------------------------------------


### 🔹 Step 2: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
DATA_PATH = "/Users/mac/Desktop/Machine Learning/DL/Homework/smile-annotations-final.csv"
MODEL_NAME = "bert-base-uncased"
OUTPUT_DIR = "./bert-smile-emotion"
HF_MODEL_NAME = "TemurbekHamzaev/bert-smile-emotion"
RANDOM_STATE = 42

BATCH_SIZE = 16
LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
MAX_LENGTH = 128

# ------------------------------------------------------------------------------


### 🔹 Step 3: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
device = torch.device(
    "mps"
    if torch.backends.mps.is_available()
    else "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Using Device:", device)


### 🔹 Step 4: Execution Block

**Purpose**: Data Ingestion and Exploration.

- Loads raw datasets into memory, inspects shape, distributions, and initial sample structures.


In [ ]:
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# ------------------------------------------------------------------------------
print("\nLoading Smile Twitter Emotion dataset...")

df = pd.read_csv(
    DATA_PATH,
    names=["id", "text", "label"],
    header=None
)


### 🔹 Step 5: Execution Block

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
df = df[["text", "label"]]

df = df.dropna(subset=["text", "label"])

df["text"] = df["text"].astype(str)
df["label"] = df["label"].astype(str)

print("Dataset Shape:", df.shape)
print("\nRaw emotion distribution:")
print(df["label"].value_counts())


### 🔹 Step 6: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
VALID_EMOTIONS = [
    "happy",
    "angry",
    "sad",
    "surprise",
    "disgust"
]

df = df[df["label"].isin(VALID_EMOTIONS)].copy()

print("\nFiltered dataset shape:", df.shape)
print("\nEmotion distribution after filtering:")
print(df["label"].value_counts())


### 🔹 Step 7: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
labels = sorted(df["label"].unique())
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

df["label"] = df["label"].map(label2id)

num_labels = len(labels)
print("\nLabel mapping:", label2id)
print("Number of classes:", num_labels)

# ------------------------------------------------------------------------------


### 🔹 Step 8: Execution Block

**Purpose**: Dataset Preprocessing & Feature Scaling.

- Splits data into Training/Validation/Testing sets to evaluate generalization.

- Standardizes features ($\mu=0, \sigma=1$) to stabilize gradient descent and prevent vanishing/exploding updates.


In [ ]:
train_df, val_df = train_test_split(
    df, 
    test_size=0.2, 
    random_state=RANDOM_STATE,
    stratify=df["label"]
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print("\nDataset split:")
print("Training samples:", len(train_df))
print("Validation samples:", len(val_df))

# ------------------------------------------------------------------------------


### 🔹 Step 9: Execution Block

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
print("\nLoading BERT tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)


class EmotionDataset(Dataset):
    Custom PyTorch Dataset for text classification.
    Tokenizes raw text samples on the fly during training/evaluation.
    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe["text"].tolist()
        self.labels = dataframe["label"].tolist()
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, index):
        text = self.texts[index]
        label = self.labels[index]


### 🔹 Step 10: Execution Block

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
encoding = self.tokenizer(
            text, 
            truncation=True,
            max_length=self.max_length
        )
        
        item = {
            "input_ids": encoding['input_ids'],
            "attention_mask": encoding['attention_mask'],
            "labels": label
        }
        
        return item


train_dataset = EmotionDataset(train_df, tokenizer, MAX_LENGTH)
val_dataset = EmotionDataset(val_df, tokenizer, MAX_LENGTH)


### 🔹 Step 11: Execution Block

**Purpose**: PyTorch Dataset & DataLoader Pipeline.

- Wraps tensors in iterable batches, handles multi-threaded worker loading, dynamic collation, and shuffling.


In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=data_collator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=data_collator
)

# ------------------------------------------------------------------------------


### 🔹 Step 12: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("\nLoading pre-trained BERT model...")

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=num_labels,
    label2id=label2id,
    id2label=id2label
)

model.to(device)


### 🔹 Step 13: Execution Block

**Purpose**: Loss Function & Optimizer Initialization.

- Configures optimization objective and update rule (e.g., Adam, SGD with momentum, weight decay).


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

total_training_steps = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_training_steps * 0.1)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)

# ------------------------------------------------------------------------------


### 🔹 Step 14: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
def train_one_epoch(model, data_loader, optimizer, scheduler, device):
    Trains the model for one epoch over the dataset.
    model.train()
    
    total_loss = 0
    all_predictions = []
    all_labels = []
    
    for batch in data_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels_batch = batch["labels"].to(device)
        
        optimizer.zero_grad()


### 🔹 Step 15: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels_batch
        )
        
        loss = outputs.loss
        logits = outputs.logits


### 🔹 Step 16: Execution Block

**Purpose**: Training & Optimization Loop.

- **Forward Pass**: Compute model predictions and loss.

- **Backward Pass**: `loss.backward()` calculates gradients via automatic differentiation.

- **Optimizer Step**: `optimizer.step()` updates trainable weights; `optimizer.zero_grad()` clears gradients.


In [ ]:
loss.backward()
        
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )
        
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()


### 🔹 Step 17: Execution Block

**Purpose**: Evaluation, Metrics & Visualization.

- Evaluates model accuracy, F1-scores, loss convergence curves, and error distributions.


In [ ]:
predictions = torch.argmax(logits, dim=-1)
        
        all_predictions.extend(predictions.detach().cpu().numpy())
        all_labels.extend(labels_batch.detach().cpu().numpy())
        
    average_loss = total_loss / len(data_loader)
        
    accuracy = accuracy_score(all_labels, all_predictions)
    f1 = f1_score(all_labels, all_predictions, average='weighted')


### 🔹 Step 18: Execution Block

**Purpose**: Evaluation, Metrics & Visualization.

- Evaluates model accuracy, F1-scores, loss convergence curves, and error distributions.


In [ ]:
return average_loss, accuracy, f1


def evaluate(model, data_loader, device):
    Evaluates model performance on the validation dataset.
    model.eval()
    
    total_loss = 0
    all_predictions = []
    all_labels = []
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels_batch = batch["labels"].to(device)
            
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels_batch
            )
            
            loss = outputs.loss
            logits = outputs.logits
            
            total_loss += loss.item()
            
            predictions = torch.argmax(logits, dim=-1)
            
            all_predictions.extend(predictions.cpu().numpy())
            all_labels.extend(labels_batch.cpu().numpy())
            
    average_loss = total_loss / len(data_loader)
    accuracy = accuracy_score(all_labels, all_predictions)
    f1 = f1_score(all_labels, all_predictions, average="weighted")
    
    return average_loss, accuracy, f1

# ------------------------------------------------------------------------------


### 🔹 Step 19: Execution Block

**Purpose**: Training & Optimization Loop.

- **Forward Pass**: Compute model predictions and loss.

- **Backward Pass**: `loss.backward()` calculates gradients via automatic differentiation.

- **Optimizer Step**: `optimizer.step()` updates trainable weights; `optimizer.zero_grad()` clears gradients.


In [ ]:
print("\nStarting BERT fine-tuning...")

best_f1 = 0.0

for epoch in range(NUM_EPOCHS):
    print(f"\nEpoch {epoch+1}/{NUM_EPOCHS}") 
    
    train_loss, train_accuracy, train_f1 = train_one_epoch(
        model, 
        train_loader, 
        optimizer, 
        scheduler, 
        device
    )
          
    val_loss, val_accuracy, val_f1 = evaluate(
        model, 
        val_loader,
        device
    )
    
    print(f"Train Loss: {train_loss:.4f}")
    print(f"Train Accuracy: {train_accuracy:.4f}")
    print(f"Train F1: {train_f1:.4f}")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_accuracy:.4f}")
    print(f"Validation F1: {val_f1:.4f}")


### 🔹 Step 20: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
if val_f1 > best_f1:
        best_f1 = val_f1
        
        os.makedirs(OUTPUT_DIR, exist_ok=True)
        
        model.save_pretrained(OUTPUT_DIR)
        tokenizer.save_pretrained(OUTPUT_DIR)
        
        print("Best Model Saved")

print("\nTraining Completed.")
print(f"Best validation F1: {best_f1:.4f}")

# ------------------------------------------------------------------------------


### 🔹 Step 21: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("\nLoading best model for final evaluation...")

model = AutoModelForSequenceClassification.from_pretrained(OUTPUT_DIR)
model.to(device)

print("\nFinal evaluation...")
val_loss, val_accuracy, val_f1 = evaluate(model, val_loader, device)

print("\nFinal validation results:")
print(f"Loss: {val_loss:.4f}")
print(f"Accuracy: {val_accuracy:.4f}")
print(f"F1-score: {val_f1:.4f}")

# ------------------------------------------------------------------------------


### 🔹 Step 22: Execution Block

**Purpose**: Core functional execution step.

- Executes the defined transformation, evaluation, or helper utility.


In [ ]:
print("\nLogging into Hugging Face...")
login()

print("\nUploading model to Hugging Face Hub...")
model.push_to_hub(HF_MODEL_NAME)
tokenizer.push_to_hub(HF_MODEL_NAME)

print("\nModel successfully uploaded.")
print(f"https://huggingface.co/{HF_MODEL_NAME}")


## 🎯 Summary & Key Takeaways
1. **Modular Execution**: Each component runs independently and validates intermediate tensor shapes and states.
2. **Core Insights**: Inspect the printed metrics, loss outputs, and visual distributions above.
3. **Next Lesson**: Applies these foundations to more advanced deep learning and transformer architectures.
